# setup


In [31]:
import os,json
from dotenv import load_dotenv
import PyPDF2
load_dotenv()

if os.environ["GROQ_API_KEY"] == "":
    print("GROQ_API_KEY is set")
# GROQ_API_KEY = os.getenv("GROQ_API_KEY", "")

In [32]:
from langchain_groq import ChatGroq
llm = ChatGroq(model="llama-3.3-70b-versatile",temperature=0,)
# print("GROQ:", "ok" if GROQ_API_KEY else "Missing add api key in .env file")

In [33]:
response =llm.invoke("What is the capital of France?")
response.content

'The capital of France is Paris.'

# loading pdf 

In [34]:
load_pdf = "docs/2023-annual-report-truncated.pdf"
reader = PyPDF2.PdfReader(load_pdf)

Pages  = [reader.pages[i].extract_text() or "" for i in range(len(reader.pages))]

print(f"PDF loaded: {len(Pages)} pages")


PDF loaded: 50 pages


In [35]:

def save_pages(pages, folder="storage/pages"):

    os.makedirs(folder, exist_ok=True)

    for i, text in enumerate(pages):

        filename = os.path.join(
            folder,
            f"page_{i+1:04}.txt"
        )

        with open(filename, "w", encoding="utf-8") as f:
            f.write(text)

    print("Pages saved.")

In [36]:
save_pages(Pages)

Pages saved.


In [37]:
def build_page_nodes(pages):
    nodes = []

    for i, page in enumerate(pages):

        node = {
            "node_id": f"P{i+1:04}",
            "level": 0,
            "title": f"Page {i+1}",
            "summary": page[:200],      # temporary summary
            "page_start": i+1,
            "page_end": i+1,
            "children": []
        }

        nodes.append(node)

    return nodes

page_nodes = build_page_nodes(Pages)
print(len(page_nodes))

50



### merge nodes

In [ ]:
def merge_nodes(nodes, level, group_size=5):

    merged = []

    count = 1

    for i in range(0, len(nodes), group_size):

        group = nodes[i:i+group_size]

        parent = generate_parent_node(group)

        merged.append({

            "node_id": f"L{level}_{count:03}",

            "level": level,

            "title": parent["title"],

            "summary": parent["summary"],

            "page_start": group[0]["page_start"],

            "page_end": group[-1]["page_end"],

            "children": group

        })

        count += 1

    return merged

# build tree

In [39]:
def build_tree(page_nodes):
    current = page_nodes
    level = 1
    while len(current) > 1:

        print(f"Level {level}: {len(current)} nodes")
        current = merge_nodes(current, level)
        level += 1

    return current[0]

tree = build_tree(page_nodes)

Level 1: 50 nodes
Level 2: 10 nodes
Level 3: 2 nodes


In [40]:
with open("storage/tree.json","w") as f:

    json.dump(tree,f,indent=4)

In [41]:
def print_tree(node, indent=0):
    print(
        "  " * indent +
        f"[{node['node_id']}] {node['title']} "
        f"(Pages {node['page_start']}-{node['page_end']})"
    )

    for child in node.get("children", []):
        print_tree(child, indent + 1)


print_tree(tree)

[L3_001] Group 1 (Pages 1-50)
  [L2_001] Group 1 (Pages 1-25)
    [L1_001] Group 1 (Pages 1-5)
      [P0001] Page 1 (Pages 1-1)
      [P0002] Page 2 (Pages 2-2)
      [P0003] Page 3 (Pages 3-3)
      [P0004] Page 4 (Pages 4-4)
      [P0005] Page 5 (Pages 5-5)
    [L1_002] Group 2 (Pages 6-10)
      [P0006] Page 6 (Pages 6-6)
      [P0007] Page 7 (Pages 7-7)
      [P0008] Page 8 (Pages 8-8)
      [P0009] Page 9 (Pages 9-9)
      [P0010] Page 10 (Pages 10-10)
    [L1_003] Group 3 (Pages 11-15)
      [P0011] Page 11 (Pages 11-11)
      [P0012] Page 12 (Pages 12-12)
      [P0013] Page 13 (Pages 13-13)
      [P0014] Page 14 (Pages 14-14)
      [P0015] Page 15 (Pages 15-15)
    [L1_004] Group 4 (Pages 16-20)
      [P0016] Page 16 (Pages 16-16)
      [P0017] Page 17 (Pages 17-17)
      [P0018] Page 18 (Pages 18-18)
      [P0019] Page 19 (Pages 19-19)
      [P0020] Page 20 (Pages 20-20)
    [L1_005] Group 5 (Pages 21-25)
      [P0021] Page 21 (Pages 21-21)
      [P0022] Page 22 (Pages 22-22)
 

In [42]:
def load_tree(path="storage/tree.json"):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)
def compress_level(children):
    result = []
    for child in children:
        result.append({
            "node_id": child["node_id"],
            "title": child["title"],
            "summary": child.get("summary", ""),
            "page_start": child.get("page_start"),
            "page_end": child.get("page_end")
        })

    return result

def choose_child(query, children):
    compact = compress_level(children)

    prompt = f"""
You are navigating a hierarchical document tree.

Your task is to choose the SINGLE child node that is most relevant to the user's question.

User Question:
{query}

Available Child Nodes:
{json.dumps(compact, indent=2)}

IMPORTANT:
- Return ONLY the node_id.
- Do NOT explain.
- Do NOT use markdown.
- Do NOT return JSON.

Example:
L1_001
"""

    response = llm.invoke(prompt)

    node_id = response.content.strip()

    print("Selected:", node_id)

    return node_id

# Traverse the Tree and pages

In [43]:
def traverse_tree(query, root):
    current = root
    path = [current["title"]]
    while current.get("children"):
        print(f"\nCurrent Node : {current['title']}")
        selected = choose_child(
            query,
            current["children"]
        )

        found = None
        for child in current["children"]:
            if child["node_id"] == selected:
                found = child
                break

        if found is None:
            print("No child selected.")
            break
        current = found
        path.append(current["title"])
    return current, path
def load_page_content(node):
    context = ""
    for page in range(
        node["page_start"],
        node["page_end"] + 1
    ):
        filename = os.path.join(
            "storage/pages",
            f"page_{page:04}.txt"
        )
        if os.path.exists(filename):

            with open(filename,
                      encoding="utf-8") as f:

                context += f.read()

                context += "\n\n"

    return context

# Generate Answer

In [44]:
def answer_query(query, context):
    prompt = f"""
Answer ONLY using the supplied context.
Question: {query}
Context: {context}
"""
    return llm.invoke(prompt).content

# final pipeline 

In [45]:
def vectorless_rag(query):

    # Load the root node
    root = load_tree()

    # Traverse the tree
    leaf, path = traverse_tree(query, root)

    print("\nTraversal Path")
    print(" -> ".join(path))

    # Load page content
    context = load_page_content(leaf)

    # Generate answer
    answer = answer_query(query, context)

    return answer

In [46]:
query = input("Ask a question: ")

print(vectorless_rag(query))


Current Node : Group 1
Selected: L2_002

Current Node : Group 2
Selected: L1_007

Current Node : Group 7
Selected: None of the provided child nodes directly mention "balance sheet policy". However, based on the context of banking and financial systems, a relevant node might not be present in the given options. Since I must choose one, I will select the first node as it discusses supervision and regulation which could be related to financial policies.

P0031
No child selected.

Traversal Path
Group 1 -> Group 2 -> Group 7
There is no mention of "balance sheet policy" in the provided context.
